# Sketch to campaign with MAI Image 2.7

This notebook turns a rough sketch into polished campaign ideas using MAI Image 2.7 and MAI Image 2.7 Flash. The flow stays intentionally small: review sketches, pick one direction, and generate a few polished scenes.

### Example: from sketch to campaign

![Example sketch, product concept and three AI-generated campaign posts](examples/sketch-to-poc-to-social.png)

MAI Image 2.7 is especially useful for product-first visuals: crisp studio lighting, believable materials, clean prompt adherence, and strong composition for campaign mockups. Use the Flash deployment for rapid exploration and the standard deployment for the final hero and lifestyle scenes.

* Run in order
* Pause to choose a concept. 
* Copy `.env.example` to `.env` and add your model endpoint and deployment names
* Your sketches in `img`
* And `az login` to your Azure/Microsoft Foundry account.


The sample setup generates 18 images, including three model-branded posts; refinements make additional API requests.

API reference.

In [ ]:
# Run once in your selected Python 3.10+ kernel; skip if already installed.
%pip install "requests>=2.31,<3" "azure-identity>=1.15,<2" "pillow>=10.1,<13" "python-dotenv>=1.0,<2"

## 1. Settings

In [ ]:
import base64
import hashlib
import io
import json
import os
import time
from contextlib import ExitStack
from datetime import datetime
from pathlib import Path
from urllib.parse import urlsplit

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import dotenv_values
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont, ImageOps

# Run from this sample folder, or from the parent folder containing it.
ROOT = Path.cwd().resolve()
if not (ROOT / "sketch-to-campaign-with-mai.ipynb").is_file():
    ROOT = ROOT / "mai-image-sample"
if not (ROOT / "sketch-to-campaign-with-mai.ipynb").is_file():
    raise ValueError("Set the notebook working directory to the sample folder or its parent.")
settings = {**dotenv_values(ROOT / ".env"), **os.environ}
ENDPOINT = (settings.get("FOUNDRY_ENDPOINT") or "").strip().rstrip("/")
CONCEPT_DEPLOYMENT = (settings.get("CONCEPT_DEPLOYMENT") or "").strip()
CAMPAIGN_DEPLOYMENT = (settings.get("CAMPAIGN_DEPLOYMENT") or "").strip()
parts = urlsplit(ENDPOINT)
if (parts.scheme != "https" or not parts.hostname or parts.path or parts.query
        or parts.fragment or parts.username or parts.password
        or parts.hostname == "your-resource.services.ai.azure.com"):
    raise ValueError("Set FOUNDRY_ENDPOINT to your HTTPS resource services endpoint.")
if not CONCEPT_DEPLOYMENT or not CAMPAIGN_DEPLOYMENT:
    raise ValueError("Set both deployment names in .env or the environment.")

SKETCH_DIR = ROOT / "img"  # Your PNG/JPEG sketches; keep originals unchanged.
REQUEST_INTERVAL = 31  # Serial requests; adjust to your assigned quota.
# To resume after a kernel restart, replace None with a previous run folder.
RUN_DIR = None
run = Path(RUN_DIR) if RUN_DIR else ROOT / "outputs" / f"campaign_{datetime.now():%Y%m%d_%H%M%S_%f}"
run.mkdir(parents=True, exist_ok=RUN_DIR is not None)
token_provider = get_bearer_token_provider(
    AzureCliCredential(), "https://cognitiveservices.azure.com/.default"
)
last_request = None
print("Outputs:", run)

In [ ]:
# Edit palette targets here. Swatches are requested colours, not measured output.
PALETTES = {
    "Warm Minimal": {"Mushroom": "#C9BFB2", "Warm white": "#F4EFE7", "Charcoal": "#303230"},
    "Green Botanical": {"Botanical green": "#355B46", "Soft sage": "#AFC7B4", "Warm ivory": "#F2F4EC"},
    "Blush Neutral": {"Dusty rose": "#CDAFA7", "Porcelain": "#F3EEE8", "Charcoal": "#343434"},
    "Architectural Dark": {"Graphite": "#383B39", "Putty": "#C7BEB0", "Nickel": "#A6A7A2"},
}
CONCEPT_BRIEF = (
    "Turn this sketch into one photorealistic pet camera and treat dispenser concept. "
    "Preserve its silhouette, proportions, camera position, treat opening and base. "
    "Smooth satin/matte finishes, dark glass lens, soft studio light, warm off-white "
    "background. Show the complete product at a three-quarter front angle. "
    "No new hardware, text, logos, people, animals or collage."
)

In [ ]:
# Helpers only validate inputs and save responses; the API calls are in the generation cells.
def prepare_image_request(references, prompt, deployment, output):
    uploads = []
    for path in references:
        with Image.open(path) as image:
            mime = {"PNG": "image/png", "JPEG": "image/jpeg"}.get(image.format)
            image.verify()
        if mime is None:
            raise ValueError(f"Use PNG/JPEG references: {path}")
        uploads.append((path, mime))
    # Match the saved request to its inputs so a new selection cannot reuse old scenes.
    spec = {
        "endpoint": ENDPOINT, "model": deployment, "prompt": prompt,
        "references": [hashlib.sha256(path.read_bytes()).hexdigest() for path, _ in uploads],
    }
    record = output.with_suffix(".json")
    if output.exists():
        if not record.is_file() or json.loads(record.read_text()) != spec:
            raise ValueError(f"Inputs changed for {output.name}; choose a fresh run folder.")
        with Image.open(output) as image:
            if image.format != "PNG":
                raise ValueError(f"Expected a saved PNG: {output}")
            image.verify()
        print("Reusing:", output.name)
        return uploads, spec, True
    return uploads, spec, False


def save_image_response(response, output, spec):
    if not response.ok:
        raise requests.HTTPError(
            f"HTTP {response.status_code}; request ID: "
            f"{response.headers.get('apim-request-id', 'not supplied')}; "
            f"Retry-After: {response.headers.get('Retry-After', 'not supplied')}. "
            f"No retry made. {response.text[:1000]}", response=response,
        )
    # The API returns base64 PNG bytes, not a local filename or URL.
    image_bytes = base64.b64decode(response.json()["data"][0]["b64_json"], validate=True)
    with Image.open(io.BytesIO(image_bytes)) as image:
        if image.format != "PNG":
            raise ValueError("The API did not return a PNG.")
        image.verify()
    output.parent.mkdir(parents=True, exist_ok=True)
    output.with_suffix(".json").write_text(json.dumps(spec, indent=2), encoding="utf-8")
    output.write_bytes(image_bytes)  # Keep the unmodified model output.

In [ ]:
# Shared local layouts: no model calls, no cropping and no platform-specific fonts.
BACKGROUND = "#F2F4EC"


def place(canvas, path, box):
    x, y, width, height = box
    with Image.open(path) as original:
        photo = ImageOps.contain(original.convert("RGB"), (width, height))
    canvas.paste(photo, (x + (width - photo.width) // 2, y + (height - photo.height) // 2))


def contact_sheet(paths, labels, output, columns=3):
    if not paths or len(paths) != len(labels):
        raise ValueError("Supply one label per image.")
    rows = (len(paths) + columns - 1) // columns
    canvas = Image.new("RGB", (columns * 400, rows * 550), BACKGROUND)
    draw = ImageDraw.Draw(canvas)
    for index, (path, label) in enumerate(zip(paths, labels)):
        x, y = index % columns * 400, index // columns * 550
        draw.text((x + 12, y + 12), label, font=ImageFont.load_default(size=17), fill="#303830")
        place(canvas, path, (x + 10, y + 50, 380, 490))
    canvas.save(output)
    display(canvas)
    return output

## 2. Sketches and concepts

Images from this run are saved in the project folder under `outputs\campaign_<timestamp>\`. The settings cell prints the exact path after **Outputs:**. Concept images, campaign scenes and branded images are saved in subfolders there; your original sketches remain in `img`.

If you set `RUN_DIR` in the settings cell, that folder is used instead.

In [ ]:
# Find PNG and JPEG sketches in the input folder, in filename order.
sketches = sorted(path for path in SKETCH_DIR.iterdir() if path.suffix.lower() in {".png", ".jpg", ".jpeg"})
# Stop early if there are no sketches to use.
if not sketches:
    raise ValueError(f"No PNG/JPEG sketches in {SKETCH_DIR}.")
# Estimate one concept per sketch and palette, plus three scenes and three branded posts.
print(f"Expected images: {len(sketches) * len(PALETTES) + 6}; refinements add more.")
# Display and save a labelled preview; this cell does not call the API.
contact_sheet(sketches, [path.name for path in sketches], run / "sketches.png")

In [ ]:
# One request per sketch/palette. Reruns reuse valid PNGs with identical inputs.
concepts = {}
for sketch in sketches:
    for palette, colours in PALETTES.items():
        colour_brief = ", ".join(f"{name} {hex_colour}" for name, hex_colour in colours.items())
        output = run / "ideas" / sketch.name / f"{palette.lower().replace(' ', '-')}.png"
        prompt = f"{CONCEPT_BRIEF}\nPalette: {colour_brief}"
        uploads, spec, reused = prepare_image_request([sketch], prompt, CONCEPT_DEPLOYMENT, output)
        if not reused:
            if last_request is not None:
                time.sleep(max(0, REQUEST_INTERVAL - (time.monotonic() - last_request)))
            token = token_provider()
            print("Generating:", output.name, flush=True)
            with ExitStack() as stack:
                files = [("image", (path.name, stack.enter_context(path.open("rb")), mime))
                         for path, mime in uploads]
                last_request = time.monotonic()
                # API request: send the sketch and prompt to the concept model.
                with requests.post(
                    f"{ENDPOINT}/mai/v1/images/edits",
                    headers={"Authorization": f"Bearer {token}"},
                    data={"model": CONCEPT_DEPLOYMENT, "prompt": prompt},
                    files=files, timeout=(15, 300),
                ) as response:
                    save_image_response(response, output, spec)
        concepts[sketch.name, palette] = output
review = contact_sheet(
    list(concepts.values()), [f"{name} | {palette}" for name, palette in concepts],
    run / "all-designs-review.png", columns=len(PALETTES),
)

## 3. Choose a design

In [ ]:
# Pause here. Choose a filename and palette from the review, not an automatic ranking.
sketch_name = "pet-3-sketch.jpg"
palette = "Blush Neutral"
source = concepts[sketch_name, palette]
with Image.open(source) as image:
    display(ImageOps.contain(image, (550, 550)))

## 4. Generate the campaign

In [ ]:
# Edit the creative direction here. Dimensions guide the image; they are not verified.
colours = ", ".join(f"{name} {value}" for name, value in PALETTES[palette].items())
campaign_brief = (
    "Create a photorealistic pet camera and treat dispenser campaign. "
    "Reference 1 defines the product: preserve its silhouette, proportions, camera, "
    f"chute and base; do not invent hardware. Palette: {colours}. "
    "Satin/matte finishes, warm natural light, dusty rose and porcelain accents, "
    "with restrained pale oak and warm stone styling that complements the blush palette. "
    "Target height 25 cm including base. Square composition, full product visible, "
    "quiet space above for copy. Treats are small brown bone-shaped baked biscuits, "
    "about 3 cm long with a narrow waist and two rounded lobes at each end. "
    "No text, logos, watermarks, app screens, collages or product performance claims."
)
SCENES = {
    "01-launch-hero": (
        "Launch hero on a low warm limestone plinth against soft porcelain and blush; gentle shadows, "
        "soft highlights, three-quarter front view matching the reference. No animals."
    ),
    "02-kitchen-lifestyle": (
        "Place the product directly on a dry continuous kitchen floor before porcelain-and-pale-oak "
        "base cabinets. Low, wide camera view; show toe-kicks and an accessible chute. "
        "No animals, countertops, shelves, platforms, sinks or water."
    ),
    "03-pet-moment": (
        "A relaxed adult tan short-haired dog with floppy ears and darker muzzle, "
        "55 cm at the shoulder, sniffs one small bone-shaped biscuit below the chute. "
        "Product LEFT facing RIGHT, dog RIGHT facing LEFT toward the front outlet. "
        "Leave a gap between muzzle and chute; keep camera and opening visible. "
        "Show whole dog, all paws and whole device on the same continuous dry floor, "
        "at the same camera depth. Device is 25 cm tall, about 45% of dog's shoulder "
        "height, well below its chest; cabinets are 90 cm tall. No forced perspective, "
        "oversized device, raised platform, treat spray or intersecting anatomy. "
        "Reference 2 guides only the kitchen floor, cabinets, light and biscuit style."
    ),
}
DOG_REFERENCE = None  # Optional PNG/JPEG path; only guides the pet's appearance.

In [ ]:
# Serial calls: every scene gets the selected concept; the pet also gets the kitchen.
campaign = {}
for name, scene in SCENES.items():
    references = [source]
    prompt = f"{campaign_brief}\nScene: {scene}"
    if name == "03-pet-moment":
        references.append(campaign["02-kitchen-lifestyle"])
        if DOG_REFERENCE is not None:
            references.append(Path(DOG_REFERENCE))
            prompt += "\nReference 3 guides ONLY the dog's appearance, not scale or layout."
    original = run / "photography" / "originals" / f"{name}.png"
    uploads, spec, reused = prepare_image_request(references, prompt, CAMPAIGN_DEPLOYMENT, original)
    if not reused:
        if last_request is not None:
            time.sleep(max(0, REQUEST_INTERVAL - (time.monotonic() - last_request)))
        token = token_provider()
        print("Generating:", original.name, flush=True)
        with ExitStack() as stack:
            files = [("image", (path.name, stack.enter_context(path.open("rb")), mime))
                     for path, mime in uploads]
            last_request = time.monotonic()
            # API request: send the reference images and scene prompt to the campaign model.
            with requests.post(
                f"{ENDPOINT}/mai/v1/images/edits",
                headers={"Authorization": f"Bearer {token}"},
                data={"model": CAMPAIGN_DEPLOYMENT, "prompt": prompt},
                files=files, timeout=(15, 300),
            ) as response:
                save_image_response(response, original, spec)
    # Preserve the original and fit the whole scene onto a square social canvas.
    output = run / "photography" / f"{name}.png"
    with Image.open(original) as image:
        ImageOps.pad(image.convert("RGB"), (1080, 1080), color=BACKGROUND).save(output)
    campaign[name] = output
    with Image.open(output) as image:
        display(ImageOps.contain(image, (550, 550)))

## 5. Generate branded posts

The model adds the text, so check the spelling and layout before sharing.

In [ ]:
# The model will be asked to add this text; it may not reproduce it exactly.
brand_name = "Treat & Peek"
tagline = "A little treat. A closer connection."
description = "A beautifully designed pet camera and treat dispenser concept for a closer connection with your pet."
labels = ["Launch Hero", "Kitchen Lifestyle", "Pet Moment"]

In [ ]:
# Ask the campaign model to add the brand copy to each scene.
if not all(text.strip() for text in (brand_name, tagline, description)):
    raise ValueError("Set the brand name, tagline and description before generating posts.")
social_dir = run / f"social_{datetime.now():%H%M%S_%f}"
social_dir.mkdir()
branded = []
for name, path in campaign.items():
    prompt = (
        "Create a polished portrait social post using the reference image. Preserve the "
        "product, scene and colour palette. Add the following text exactly as written, "
        "with clear, legible typography that does not obscure the product. "
        "Do not add any other text or logos.\n"
        f"Brand name: {brand_name}\nTagline: {tagline}\nDescription: {description}"
    )
    original = social_dir / "originals" / f"{name}-branded.png"
    uploads, spec, reused = prepare_image_request([path], prompt, CAMPAIGN_DEPLOYMENT, original)
    if not reused:
        if last_request is not None:
            time.sleep(max(0, REQUEST_INTERVAL - (time.monotonic() - last_request)))
        token = token_provider()
        print("Generating:", original.name, flush=True)
        with ExitStack() as stack:
            files = [("image", (reference.name, stack.enter_context(reference.open("rb")), mime))
                     for reference, mime in uploads]
            last_request = time.monotonic()
            # API request: send the scene and brand copy to the campaign model.
            with requests.post(
                f"{ENDPOINT}/mai/v1/images/edits",
                headers={"Authorization": f"Bearer {token}"},
                data={"model": CAMPAIGN_DEPLOYMENT, "prompt": prompt},
                files=files, timeout=(15, 300),
            ) as response:
                save_image_response(response, original, spec)
    # Keep the model output and fit it to the social-post size without cropping.
    output = social_dir / f"{name}-branded.png"
    with Image.open(original) as image:
        ImageOps.pad(image.convert("RGB"), (1080, 1350), color=BACKGROUND).save(output)
    branded.append(output)
    with Image.open(output) as image:
        display(ImageOps.contain(image, (550, 690)))
sheet = contact_sheet(branded, labels, run / "all-social-campaign-assets.png")

In [ ]:
# Local recap: the original sketch, the selected render, then the three branded posts.
journey = contact_sheet(
    [SKETCH_DIR / sketch_name, source, *branded],
    ["Original sketch", f"Selected: {palette}", *labels],
    run / "sketch-to-poc-to-social.png", columns=5,
)
print("Saved campaign:", run)
# Before sharing: review geometry, colours, pet anatomy, rights and claims.
# Label visuals AI-generated concepts, and clear notebook outputs before publishing.

## What we created

We used MAI Image 2.7 Flash to explore product concepts from sketches, then MAI Image 2.7 to create campaign scenes and branded social posts. Python and Pillow helped us preview, arrange and save the images, including a sketch-to-campaign recap.

## Next step

In Microsoft Foundry, open **Build > Catalog > Models**, select an MAI image model and choose **Deploy**. Use its deployment name and resource endpoint in your `.env` file. See Deploy and use MAI image models for prerequisites, deployment and image-editing examples.